In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_24_try_0.pickle

In [ ]:
%%cudf.pandas.profile
### cell 25 ###

def grab_subset_of_data_37(original_df, question_of_interest):
    # GPU-accelerated column filtering
    new_df = original_df.filter(like=question_of_interest)
    # build a small rename mapping on CPU; strip both leading/trailing whitespace
    mapping = {col: col.split('-')[-1].strip() for col in new_df.columns}
    # rename and drop rows that are all null in the subset
    new_df = new_df.rename(columns=mapping).dropna(how='all')
    return new_df

question_of_interest_cell37 = (
    "Which of the following integrated development environments (IDE's) "
    "do you use on a regular basis?"
)
ide_df_2022_cell37 = grab_subset_of_data_37(
    responses_df_2022_cell10, question_of_interest_cell37
)

# define groups of columns to combine (names now have no trailing spaces)
jupyter_cols = ['Jupyter Notebook', 'JupyterLab']
vs_cols      = ['Visual Studio Code (VSCode)', 'Visual Studio']

# combine Jupyter columns into one GPU operation
ide_df_2022_cell37['Jupyter/JupyterLab'] = (
    ide_df_2022_cell37[jupyter_cols]
      .any(axis=1)
      .replace({True: 'Jupyter/JupyterLab', False: np.nan})
)

# combine Visual Studio columns into one GPU operation
ide_df_2022_cell37['Visual Studio / Visual Studio Code (VSCode)'] = (
    ide_df_2022_cell37[vs_cols]
      .any(axis=1)
      .replace({True: 'Visual Studio / Visual Studio Code (VSCode)', False: np.nan})
)

# drop the now-redundant individual columns in a single GPU operation
ide_df_2022_cell37 = ide_df_2022_cell37.drop(columns=jupyter_cols + vs_cols)

ide_df_2022_cell37.info()